In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
project_path = Path("/content/drive/MyDrive/ISL_GAN_FewShot_Project")

landmark_dir = project_path / "data" / "landmarks"

train_csv = landmark_dir / "train_landmarks.csv"
val_csv   = landmark_dir / "val_landmarks.csv"
test_csv  = landmark_dir / "test_landmarks.csv"

mapping_file = landmark_dir / "class_mapping.json"

model_dir = project_path / "models"
model_dir.mkdir(parents=True, exist_ok=True)

print("Landmark directory:", landmark_dir)
print("Model directory:", model_dir)

Landmark directory: /content/drive/MyDrive/ISL_GAN_FewShot_Project/data/landmarks
Model directory: /content/drive/MyDrive/ISL_GAN_FewShot_Project/models


In [9]:
with open(mapping_file, "r") as f:
    mapping = json.load(f)

class_to_idx = mapping["class_to_idx"]
idx_to_class = {int(k): v for k, v in mapping["idx_to_class"].items()}

num_classes = len(class_to_idx)
print("Number of classes:", num_classes)
print("Classes:", list(class_to_idx.keys()))

Number of classes: 35
Classes: ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [10]:
train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (170, 66)
Val shape: (6236, 66)
Test shape: (6223, 66)


In [11]:
feature_cols = [c for c in train_df.columns if c.startswith("f")]

X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df["class_name"].map(class_to_idx).values.astype(np.int64)

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df["class_name"].map(class_to_idx).values.astype(np.int64)

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df["class_name"].map(class_to_idx).values.astype(np.int64)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (170, 63)
y_train: (170,)


In [12]:
class LandmarkDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [13]:
BATCH_SIZE = 32

train_dataset = LandmarkDataset(X_train, y_train)
val_dataset   = LandmarkDataset(X_val, y_val)
test_dataset  = LandmarkDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Dataloaders created successfully.")

Dataloaders created successfully.


In [14]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BaselineMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]

model = BaselineMLP(input_dim=input_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Model initialized on:", device)

Model initialized on: cuda


In [16]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, all_labels, all_preds

In [17]:
EPOCHS = 30

train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_val_acc = 0.0
best_model_path = model_dir / "baseline_mlp_best.pth"

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)

print("Best validation accuracy:", best_val_acc)
print("Best model saved at:", best_model_path)

Epoch [1/30] Train Loss: 3.5568, Train Acc: 0.0412 | Val Loss: 3.5430, Val Acc: 0.0693
Epoch [2/30] Train Loss: 3.5382, Train Acc: 0.0647 | Val Loss: 3.5310, Val Acc: 0.1086
Epoch [3/30] Train Loss: 3.5276, Train Acc: 0.0588 | Val Loss: 3.5176, Val Acc: 0.1270
Epoch [4/30] Train Loss: 3.5147, Train Acc: 0.1059 | Val Loss: 3.5000, Val Acc: 0.1490
Epoch [5/30] Train Loss: 3.4836, Train Acc: 0.1059 | Val Loss: 3.4762, Val Acc: 0.1801
Epoch [6/30] Train Loss: 3.4597, Train Acc: 0.1235 | Val Loss: 3.4434, Val Acc: 0.1982
Epoch [7/30] Train Loss: 3.4301, Train Acc: 0.1706 | Val Loss: 3.3988, Val Acc: 0.1554
Epoch [8/30] Train Loss: 3.3746, Train Acc: 0.1353 | Val Loss: 3.3410, Val Acc: 0.1397
Epoch [9/30] Train Loss: 3.3066, Train Acc: 0.1471 | Val Loss: 3.2684, Val Acc: 0.1786
Epoch [10/30] Train Loss: 3.2442, Train Acc: 0.1235 | Val Loss: 3.1830, Val Acc: 0.1974
Epoch [11/30] Train Loss: 3.1361, Train Acc: 0.1353 | Val Loss: 3.0860, Val Acc: 0.2136
Epoch [12/30] Train Loss: 3.0507, Train A